# 08.0 音频生成数据管线：清洗、切片与增强

音频生成模型训练使用的是成对样本：目标音频，以及描述生成目标的条件，例如文本 prompt、歌词、类别、乐器、风格标签或音频前缀。数据管线的任务是把原始音频整理成稳定、可追溯、可批处理的训练样本。

本 Notebook 用配套音频包演示一条小规模但完整的流程：资产检查、质量报告、重采样、转单声道、静音裁剪、RMS/peak 归一化、固定窗口切片、按目的组织的数据增强，以及后续训练/微调会读取的 manifest。增强是服务于训练目标的策略组合；manifest 会记录每个增强样本的 `policy`、`transform` 和参数说明。


## 0. 路径、依赖与辅助函数

本章代码从当前目录向上自动定位 `CODE/chapter08`（在仓库内任意目录启动均可）。输出音频写入 `output_audio/08_0`，表格写入 `outputs/tables`，图写入 `output_figures`。


In [ ]:
from pathlib import Path
import sys

# 路径推断：从 cwd 向上找含 CODE/chapter08/_common 的目录；ROOT 指向 CODE/chapter08/
_p = Path.cwd()
while not (_p / "CODE" / "chapter08" / "_common").exists():
    _parent = _p.parent
    if _parent == _p:
        raise FileNotFoundError("未找到项目根目录（包含 CODE/chapter08/_common 的目录），请在项目内运行本 Notebook")
    _p = _parent
ROOT = _p / "CODE" / "chapter08"
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import Audio, display

from _common.audio_io import load_audio
from _common.dataset_registry import asset_status
from _common.paths import portable_path
from _common.plotting import setup_plot_style
from _common.tables import write_rows
from preprocessing.audio_dataset_pipeline import (
    QUALITY_FIELDS,
    audio_signal_stats,
    prepare_author_audio_dataset,
    plot_preprocessing_summary,
    training_data_usage_rows,
)

OUTPUT_AUDIO = ROOT / "output_audio" / "08_0"
OUTPUT_FIGURES = ROOT / "output_figures"
OUTPUT_TABLES = ROOT / "outputs" / "tables"
AUTHOR_MANIFEST = ROOT / "data_manifests" / "audio_author_ch08.csv"
AUTHOR_EXAMPLE_MANIFEST = ROOT / "data_manifests" / "audio_author_ch08.example.csv"
AUTHOR_AUDIO_ROOT = ROOT.parent / "datasets" / "audio_author" / "chapter_08_author"
for path in [OUTPUT_AUDIO, OUTPUT_FIGURES, OUTPUT_TABLES]:
    path.mkdir(parents=True, exist_ok=True)
setup_plot_style()

def rel(path):
    return portable_path(path, ROOT)

def resolve_output_path(path_text):
    path = Path(path_text)
    if path.is_absolute():
        return path
    for base in [ROOT, ROOT.parent, ROOT.parents[1]]:
        candidate = base / path
        if candidate.exists():
            return candidate
    return ROOT / path

def pick_augmentation_row(augmentation_df, source_case_id, preferred_transform):
    rows = augmentation_df[augmentation_df["source_case_id"] == source_case_id]
    preferred = rows[rows["transform"] == preferred_transform]
    if not preferred.empty:
        return preferred.iloc[0]
    return rows.iloc[0] if not rows.empty else None

def signal_diagnostic_row(label, path_text, policy="", transform=""):
    path = resolve_output_path(path_text)
    audio, sr = load_audio(path, sr=None, mono=True)
    stats = audio_signal_stats(audio)
    return {
        "label": label,
        "policy": policy,
        "transform": transform,
        "path": rel(path),
        "sample_rate": sr,
        "duration_sec": round(len(audio) / sr, 3) if sr else 0.0,
        "peak_dbfs": round(stats["peak_dbfs"], 2),
        "rms_dbfs": round(stats["rms_dbfs"], 2),
    }

def display_audio_without_normalization(path_text, label):
    path = resolve_output_path(path_text)
    audio, sr = load_audio(path, sr=None, mono=True)
    print(label)
    print(rel(path))
    display(Audio(audio, rate=sr, normalize=False))


## 1. 资产检查与 manifest

训练脚本读取 manifest，不直接扫描任意 wav 文件夹。manifest 记录音频路径、划分、类别、文本条件和授权状态。资产缺失时，本单元打印下载提示；准备好数据后重新运行即可。


In [ ]:
asset = asset_status("audio_author_ch08")
asset_row = {
    "asset_id": asset.spec.asset_id,
    "available": asset.ok,
    "paths": ";".join(rel(path) for path in asset.existing_paths),
    "download_hint": asset.spec.download_hint,
    "manifest": rel(AUTHOR_MANIFEST),
}
write_rows(OUTPUT_TABLES / "08_0_author_asset_status.csv", [asset_row])
display(pd.DataFrame([asset_row]))
if not asset.ok:
    print(asset.spec.download_hint)
if not AUTHOR_MANIFEST.exists():
    print("Missing author manifest:", rel(AUTHOR_MANIFEST))
    print("Use this schema and fill real rows:")
    display(pd.read_csv(AUTHOR_EXAMPLE_MANIFEST))


## 2. 质量检查、清理与切片

质量检查先读取文件头和信号统计量，给出 keep/reject 决策。清理阶段统一采样率和声道，裁剪长静音，把 RMS 调到稳定范围，再限制峰值。切片阶段把变长音频切成固定长度窗口，让 WaveNet、codec token 缓存、MusicGen 微调和 LoRA 训练都能稳定组 batch。


In [ ]:
ready = asset.ok and AUTHOR_MANIFEST.exists()
if ready:
    result = prepare_author_audio_dataset(
        manifest_csv=AUTHOR_MANIFEST,
        audio_root=AUTHOR_AUDIO_ROOT,
        output_audio_dir=OUTPUT_AUDIO,
        output_table_dir=OUTPUT_TABLES,
        chapter_root=ROOT,
        target_sr=32000,
        segment_sec=8.0,
        hop_sec=8.0,
        max_files=3,
        max_segments_per_file=2,
    )
    quality_df = pd.DataFrame(result["quality"])
    segment_df = pd.DataFrame(result["segments"])
    augmentation_df = pd.DataFrame(result["augmentations"])
else:
    result = {
        "quality": [
            {
                "case_id": "audio_author_ch08_missing",
                "source_path": rel(AUTHOR_AUDIO_ROOT),
                "status": "missing_asset",
                "decision": "download_required",
                "sample_rate": "",
                "channels": "",
                "duration_sec": "",
                "peak_dbfs": "",
                "rms_dbfs": "",
                "clipping_ratio": "",
                "silence_ratio": "",
                "prompt_en": "",
                "prompt_cn": "",
                "detail": asset.spec.download_hint,
            }
        ],
        "segments": [],
        "augmentations": [],
        "training_usage": training_data_usage_rows(),
    }
    write_rows(OUTPUT_TABLES / "08_0_author_audio_quality.csv", result["quality"], fieldnames=QUALITY_FIELDS)
    quality_df = pd.DataFrame(result["quality"])
    segment_df = pd.DataFrame()
    augmentation_df = pd.DataFrame()

display(quality_df)
if not segment_df.empty:
    display(segment_df.head(12))
else:
    print("No prepared segments yet. Download the author audio pack and rerun this Notebook.")


## 3. 按目的组织数据增强

小数据训练常把几类增强组合使用：电平扰动处理响度差异，噪声增强处理录音环境差异，轻微移调/变速扩展局部音乐变化，低通和噪声模拟传输或 codec 退化。每种增强都有边界：如果任务条件包含精确音高、速度、歌词对齐或演奏版本，相关增强要从策略中移除。

下面的表格展示本 Notebook 导出的增强 manifest。`policy` 表示训练目的，`transform` 表示具体信号处理步骤。


In [ ]:
if not augmentation_df.empty:
    display(augmentation_df.head(20))
else:
    print("No augmentation rows yet.")


## 4. 听感比较与信号诊断

轻微电平变化常被播放器、系统音量或人的响度感知弱化。这里播放音频数组时显式设置 `normalize=False`，并同时显示 peak/RMS。电平类增强看 RMS；组合增强用于观察音高、速度、噪声或频带变化。


In [ ]:
selected_aug_row = None
gain_aug_row = None
if not segment_df.empty and not augmentation_df.empty:
    first_segment_row = segment_df.iloc[0]
    source_case_id = first_segment_row["case_id"]
    selected_aug_row = pick_augmentation_row(
        augmentation_df,
        source_case_id=source_case_id,
        preferred_transform="combo_pitch_tempo_gain_noise",
    )
    gain_aug_row = pick_augmentation_row(
        augmentation_df,
        source_case_id=source_case_id,
        preferred_transform="gain_minus_6db",
    )

    diagnostic_rows = [
        signal_diagnostic_row("clean_segment", first_segment_row["path"], transform="clean"),
    ]
    if gain_aug_row is not None:
        diagnostic_rows.append(
            signal_diagnostic_row(
                "gain_example",
                gain_aug_row["path"],
                policy=gain_aug_row["policy"],
                transform=gain_aug_row["transform"],
            )
        )
    if selected_aug_row is not None and (
        gain_aug_row is None or selected_aug_row["path"] != gain_aug_row["path"]
    ):
        diagnostic_rows.append(
            signal_diagnostic_row(
                "combined_example",
                selected_aug_row["path"],
                policy=selected_aug_row["policy"],
                transform=selected_aug_row["transform"],
            )
        )
    display(pd.DataFrame(diagnostic_rows))

    display_audio_without_normalization(first_segment_row["path"], "Clean prepared segment")
    if gain_aug_row is not None:
        display_audio_without_normalization(gain_aug_row["path"], "Gain augmentation, no playback normalization")
    if selected_aug_row is not None and (
        gain_aug_row is None or selected_aug_row["path"] != gain_aug_row["path"]
    ):
        display_audio_without_normalization(selected_aug_row["path"], "Combined augmentation, no playback normalization")
else:
    print("No prepared segments yet. Download the author audio pack and rerun this Notebook.")


## 5. 可视化清理与增强效果

图中比较同一段素材的三种形态：原始片段、清洗后的训练片段和一个组合增强片段。


In [ ]:
if not segment_df.empty and not augmentation_df.empty:
    first_segment_row = segment_df.iloc[0]
    if selected_aug_row is None:
        selected_aug_row = pick_augmentation_row(
            augmentation_df,
            source_case_id=first_segment_row["case_id"],
            preferred_transform="combo_pitch_tempo_gain_noise",
        )
    source_path = resolve_output_path(first_segment_row["source_path"])
    cleaned_path = resolve_output_path(first_segment_row["path"])
    augmented_path = resolve_output_path(selected_aug_row["path"])
    fig = plot_preprocessing_summary(
        source_path=source_path,
        cleaned_path=cleaned_path,
        augmented_path=augmented_path,
        out_path=OUTPUT_FIGURES / "08_0_audio_preprocessing_summary.png",
    )
    plt.show()
else:
    print("Skip preprocessing figure because no prepared segment is available.")


## 6. 本管线产物如何进入后续模型

质量表用于筛选样本，切片 manifest 用于训练或 token 缓存，增强 manifest 用于小数据微调和鲁棒性实验。后续 Notebook 读取这些 CSV，而不是重新实现数据清理逻辑。


In [ ]:
usage_df = pd.DataFrame(result["training_usage"])
display(usage_df)
print("08_0 outputs:")
for path in [
    OUTPUT_TABLES / "08_0_author_asset_status.csv",
    OUTPUT_TABLES / "08_0_author_audio_quality.csv",
    OUTPUT_TABLES / "08_0_prepared_segments.csv",
    OUTPUT_TABLES / "08_0_augmentation_manifest.csv",
    OUTPUT_TABLES / "08_0_training_data_usage.csv",
    OUTPUT_FIGURES / "08_0_audio_preprocessing_summary.png",
]:
    print("-", rel(path), "exists=", path.exists())
